# Projet IC1 (2/3) — Modèles, validation par sondage, seuil de coût

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_Projet_S4_modeles_couts.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['composites_2m.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
comp = pd.read_csv('composites_2m.csv'); comp['mineralise'] = (comp['au_gpt'] > 0.5).astype(int)
num, cat = ['sulfures_pct', 'alteration', 'de_m'], ['lithologie']
y, g = comp['mineralise'], comp['sondage']
prep = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])

## 1. Comparer trois modèles : k-fold aléatoire vs par sondage

In [ ]:
modeles = {'logit': LogisticRegression(max_iter=2000), 'rf': RandomForestClassifier(400, min_samples_leaf=5, random_state=0),
           'gb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, random_state=0)}
for nom, est in modeles.items():
    pipe = Pipeline([('prep', prep), ('m', est)])
    a = cross_val_score(pipe, comp[num + cat], y, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc').mean()
    b = cross_val_score(pipe, comp[num + cat], y, cv=GroupKFold(5), groups=g, scoring='roc_auc').mean()
    print(nom, round(a, 3), round(b, 3))

## 2. Et si l’on ajoute X, Y ? Que signifie le gain ?

In [ ]:
prep_xy = ColumnTransformer([('num', StandardScaler(), num + ['x_m', 'y_m']), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
pipe = Pipeline([('prep', prep_xy), ('m', RandomForestClassifier(400, min_samples_leaf=5, random_state=0))])
print(cross_val_score(pipe, comp[num + cat + ['x_m', 'y_m']], y, cv=GroupKFold(5), groups=g, scoring='roc_auc').mean())

## 3. Seuil optimal (coût FN = 2 × FP)
À MODIFIER : le rapport de coûts, après discussion avec le « géologue chef ».

In [ ]:
gb = Pipeline([('prep', prep), ('m', modeles['gb'])])
p = cross_val_predict(gb, comp[num + cat], y, cv=GroupKFold(5), groups=g, method='predict_proba')[:, 1]
print('AUC :', round(roc_auc_score(y, p), 3))
ratio = 2
seuils = np.linspace(0.02, 0.98, 97)
cout = np.array([((p >= t) & (y == 0)).sum() + ratio * ((p < t) & (y == 1)).sum() for t in seuils])
t_opt = seuils[cout.argmin()]
print('seuil optimal', round(t_opt, 2), 'coût', cout.min(), ' | coût à 0,5 :', cout[48])
plt.plot(seuils, cout); plt.axvline(t_opt, color='r'); plt.xlabel('seuil'); plt.ylabel('coût'); plt.show()